📋 Cell 1: Setup and Imports
This cell imports all the libraries we'll need for all parts of the project, including TensorFlow, Scikit-learn, and utilities.

In [41]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import glob
import joblib
import random
import cv2
import warnings
import argparse
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix, 
    classification_report, f1_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess
from tensorflow.keras.applications.imagenet_utils import preprocess_input as imagenet_preprocess

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print("TensorFlow Version:", tf.__version__)
print("All libraries imported successfully.")

TensorFlow Version: 2.18.0
All libraries imported successfully.


⚙️ Cell 2: Hardware Detection
This utility function (from your deepfake-detection2 notebook) checks for available hardware (GPU/TPU) and sets the appropriate TensorFlow distribution strategy.

In [42]:
def get_distribution_strategy():
    """
    Detects available hardware (TPU, multi-GPU, single-GPU, CPU) and returns
    the appropriate TensorFlow distribution strategy.
    """
    try:
        # Attempt to detect and initialize a TPU
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect()
        strategy = tf.distribute.TPUStrategy(tpu)
        print("✅ Running on TPU")
    except (ValueError, tf.errors.NotFoundError):
        # If no TPU is found, check for GPUs
        gpus = tf.config.list_physical_devices('GPU')
        if len(gpus) > 1:
            # If multiple GPUs are available, use MirroredStrategy
            strategy = tf.distribute.MirroredStrategy()
            print(f"✅ Running on {len(gpus)} GPUs")
        elif len(gpus) == 1:
            # If a single GPU is available, use the default strategy
            strategy = tf.distribute.get_strategy()
            print("✅ Running on a single GPU")
        else:
            # If no GPUs are found, run on CPU
            strategy = tf.distribute.get_strategy()
            print("✅ Running on CPU")
            
    print(f"Number of accelerator replicas: {strategy.num_replicas_in_sync}")
    return strategy

# Run the detection function
strategy = get_distribution_strategy()

✅ Running on a single GPU
Number of accelerator replicas: 1


📁 Cell 3: Writing Project Files (Models & Utils)
Here we write the Python scripts for our models and utilities. This is the modular approach you requested.

Cell 3a: model_attention.py
This is the model architecture from your deepfake-detection2 notebook. I've added a name="attention_output" to the Attention layer, which is critical for extracting its features later, as specified in your Plan.txt.

In [43]:
%%writefile model_attention.py
import tensorflow as tf

def backbone():
    '''
    RETURNS THE BACKBONE FEATURE ENCODER NETWORK
    XCEPTION USED IN THIS CASE
    '''
    mod  = tf.keras.applications.Xception(weights='imagenet')
    mod = tf.keras.Model(mod.input, mod.layers[-13].output)
    return mod
    
class ModifiedBranch(tf.keras.layers.Layer):
    '''
    COMPUTES THE MODIFIED BRANCH TO BE USED IN ATTENTION TECHNIQUE
    '''
    def __init__(self, a_vec_size, **kwargs):
        super(ModifiedBranch, self).__init__(**kwargs)
        self.a_vec_size = a_vec_size

    def build(self, input_shape):
        self.dense_layer = tf.keras.layers.Dense(self.a_vec_size, activation='tanh')

    def call(self, input):
        af = tf.keras.backend.mean(input, axis=2) 
        hs = self.dense_layer(af)
        return hs

class MainBranch(tf.keras.layers.Layer):
    def __init__(self, a_vec_size, dim, **kwargs):
        super(MainBranch, self).__init__(**kwargs)
        self.a_vec_size = a_vec_size
        self.dim = dim

    def build(self, input_shape):
        self.reshape1 = tf.keras.layers.Reshape((-1, self.a_vec_size))
        self.relu = tf.keras.activations.relu
        self.dropout = tf.keras.layers.Dropout(0.5)
        self.reshape2 = tf.keras.layers.Reshape((self.dim**2, self.a_vec_size))

    def call(self, input):
        e = tf.transpose(input, perm=[0, 2, 1])
        e = self.reshape1(e)
        e = self.relu(e)
        e = self.dropout(e)
        e = self.reshape2(e)
        e = tf.transpose(e, perm=[0, 2, 1])
        return e

class Attention(tf.keras.layers.Layer):
    '''
    IMPLEMENTATION OF THE ATTENTION TECHNIQUE ON TWO BRANCHES
    '''
    def __init__(self, dim, a_vec_size, **kwargs):
        super(Attention, self).__init__(**kwargs)
        self.dim = dim
        self.a_vec_size = a_vec_size
    
    def build(self, input_shape):
        self.dense1 = tf.keras.layers.Dense(self.dim**2)
        self.reshape1 = tf.keras.layers.Reshape((1, self.dim**2))
        self.add = tf.keras.layers.Add()
        self.dropout = tf.keras.layers.Dropout(0.5)
        self.relu = tf.keras.activations.relu
        self.reshape2 = tf.keras.layers.Reshape((-1, self.a_vec_size))
        self.dense2 = tf.keras.layers.Dense(1, use_bias=False)
        self.reshape3 = tf.keras.layers.Reshape((-1, self.dim**2))

    def call(self, input):
        eh = self.dense1(input[0])
        eh = self.reshape1(eh)
        eh = self.add([input[1], eh])
        eh = self.relu(eh)
        eh = self.dropout(eh)
        eh = tf.transpose(eh, perm=[0, 2, 1])
        eh = self.reshape2(eh)
        eh = self.dense2(eh)
        eh = self.reshape3(eh)
        eh = self.relu(eh)
        return eh

def model(a_vec_size, dim):
    '''
    THIS FUNCTION CALLS THE ENTIRE MODEL
    '''
    back = backbone()
    backbone_feature = back.output  
    out = tf.keras.layers.Conv2D(filters = a_vec_size, kernel_size = (1,1), strides=(1,1), padding = 'valid', use_bias=True)(backbone_feature)
    out = tf.keras.layers.BatchNormalization(axis=-1)(out)
    out = tf.keras.activations.relu(out)
    out = tf.keras.layers.Dropout(0.8)(out)
    out = tf.keras.layers.Reshape((a_vec_size, dim**2))(out)
    
    modified = ModifiedBranch(a_vec_size)(out)
    main = MainBranch(a_vec_size, dim)(out)
    
    # --- CRITICAL MODIFICATION from Plan.txt: Added name='attention_output' ---
    att = Attention(dim, a_vec_size, name="attention_output")([modified, main])
    
    fin = tf.keras.layers.Dense(2, activation='softmax')(att)
    fin = tf.keras.layers.Flatten()(fin)
    mod = tf.keras.Model(inputs=back.input, outputs=fin)
    return mod

Overwriting model_attention.py


Cell 3b: utils_hybrid.py
This is our new, combined utility script. It handles data loading for your specific folder structure and includes the two different preprocessing functions we need.

In [44]:
%%writefile utils_hybrid.py
import os
import glob
import numpy as np
import cv2
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess
from tensorflow.keras.applications.imagenet_utils import preprocess_input as imagenet_preprocess
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle

# --- Data Loading ---
def get_files_from_structure(data_path):
    """
    Gets file paths and labels from the train/val/test structure.
    Label: 0 = real, 1 = fake
    """
    fake_paths = glob.glob(os.path.join(data_path, 'fake', '*.png'))
    real_paths = glob.glob(os.path.join(data_path, 'real', '*.png'))
    
    file_paths = fake_paths + real_paths
    labels = [1] * len(fake_paths) + [0] * len(real_paths)
    
    file_paths, labels = shuffle(file_paths, labels, random_state=42)
    
    print(f"Found {len(file_paths)} images in {data_path}: {len(fake_paths)} fake, {len(real_paths)} real.")
    return file_paths, labels

# --- Image Preprocessing ---
def load_and_prep_image(path, target_size, preprocess_type='xception'):
    """
    Loads and preprocesses a single image.
    preprocess_type can be 'xception' or 'imagenet'
    """
    try:
        img = cv2.imread(path)
        if img is None:
            print(f"Warning: Could not read image {path}. Skipping.")
            return None
        
        img_resized = cv2.resize(img, target_size)
        
        if preprocess_type == 'xception':
            img_preprocessed = xception_preprocess(img_resized)
        else: # 'imagenet'
            img_preprocessed = imagenet_preprocess(img_resized)
            
        return img_preprocessed
    except Exception as e:
        print(f"Error processing image {path}: {e}")
        return None

# --- Keras Data Generators ---
def image_generator(file_paths, labels, batch_size, target_size=(299, 299), preprocess_type='xception'):
    """
    Keras generator for training models.
    """
    num_samples = len(file_paths)
    while True:
        file_paths, labels = shuffle(file_paths, labels)
        
        for offset in range(0, num_samples, batch_size):
            batch_paths = file_paths[offset:offset+batch_size]
            batch_labels = labels[offset:offset+batch_size]
            
            batch_x = []
            batch_y = []
            
            for i, input_path in enumerate(batch_paths):
                img = load_and_prep_image(input_path, target_size, preprocess_type)
                if img is not None:
                    batch_x.append(img)
                    batch_y.append(batch_labels[i])
            
            if batch_x:
                batch_x = np.array(batch_x)
                batch_y = to_categorical(np.array(batch_y), num_classes=2)
                yield batch_x, batch_y

Overwriting utils_hybrid.py


🏛️ Cell 4: Part 1 - Train Model 1 (Xception+Attention)
First, we'll write and run the training script for the Xception+Attention model. This will create best_model_attention.keras.

Cell 4a: train_attention.py

In [45]:
%%writefile train_attention.py
import os
import argparse
import tensorflow as tf
from model_attention import model, ModifiedBranch, MainBranch, Attention
from utils_hybrid import get_files_from_structure, image_generator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# --- Constants based on your data ---
# Train: 6028 fake + 5605 real = 11633
# Val: 1200 fake + 1200 real = 2400
BATCH_SIZE = 32
TRAIN_STEPS = 11633 // BATCH_SIZE
VAL_STEPS = 2400 // BATCH_SIZE
IMG_DIM = (299, 299)
A_VEC_SIZE = 1024 # From model architecture
DIM = 19         # From model architecture

def train(args):
    # Load data
    train_files, train_labels = get_files_from_structure(args.train_path)
    val_files, val_labels = get_files_from_structure(args.val_path)
    
    # Create generators
    train_gen = image_generator(train_files, train_labels, BATCH_SIZE, IMG_DIM, 'xception')
    val_gen = image_generator(val_files, val_labels, BATCH_SIZE, IMG_DIM, 'xception')
    
    # Get strategy
    # Note: get_distribution_strategy() is in the main notebook, 
    # so for this script, we'll use the default.
    # For multi-GPU, you'd import and call it here inside a 'with strategy.scope()':
    
    mod = model(A_VEC_SIZE, DIM)
    mod.compile('Adam', loss=tf.keras.losses.CategoricalCrossentropy(), metrics=['accuracy'])
    
    print("************ TRAINING XCEPTION+ATTENTION MODEL ************")
    
    checkpoint_filepath = os.path.join(args.output_path, "best_model_attention.keras")
    model_checkpoint_callback = ModelCheckpoint(
        filepath=checkpoint_filepath, 
        save_best_only=True, 
        monitor='val_accuracy', 
        mode='max',
        verbose=1
    )
    
    early_stopping_callback = EarlyStopping(
        monitor='val_accuracy',
        patience=5, # Stop after 5 epochs of no improvement
        restore_best_weights=True
    )

    mod.fit(train_gen, 
              epochs=args.epochs, 
              steps_per_epoch=TRAIN_STEPS,
              validation_data=val_gen,
              validation_steps=VAL_STEPS,
              callbacks=[model_checkpoint_callback, early_stopping_callback])
    
    print(f"Finished training. Best model saved to {checkpoint_filepath}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description='Train Xception+Attention Model')
    parser.add_argument('--train_path', type=str, required=True, help='Path to training data folder')
    parser.add_argument('--val_path', type=str, required=True, help='Path to validation data folder')
    parser.add_argument('--output_path', type=str, default="/kaggle/working/", help='Path to save model')
    parser.add_argument('--epochs', type=int, default=3, help='Number of epochs')
    
    args = parser.parse_args()
    
    os.makedirs(args.output_path, exist_ok=True)
    train(args)

Overwriting train_attention.py


Cell 4b: Run Attention Model Training
Now, we execute the script. This will take some time. It will train the model and save the best checkpoint based on val_accuracy.

In [46]:
!python train_attention.py \
    --train_path "/kaggle/input/1000-videos-split/1000_videos/train" \
    --val_path "/kaggle/input/1000-videos-split/1000_videos/validation" \
    --output_path "/kaggle/working/models/" \
    --epochs 30

2025-11-15 14:02:51.539033: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763215371.560563  217453 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763215371.567012  217453 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
Found 11633 images in /kaggle/input/1000-videos-split/1000_videos/train: 6028

🏛️ Cell 5: Part 2 - Train Model 2 (Backbones)
Next, we train the DenseNet121 and EfficientNetB0 models from your second project. These will be saved as separate .keras files.

Cell 5a: train_backbones.py

In [47]:
%%writefile train_backbones.py
import os
import argparse
import tensorflow as tf
from utils_hybrid import get_files_from_structure, image_generator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# --- Constants based on your data ---
BATCH_SIZE = 32
TRAIN_STEPS = 11633 // BATCH_SIZE
VAL_STEPS = 2400 // BATCH_SIZE
IMG_DIM = (299, 299)
INPUT_SHAPE = (299, 299, 3)

def create_fine_tuned_model(model_name, num_classes=2):
    """Create and fine-tune pre-trained models"""
    if model_name == 'DenseNet121':
        base_model = tf.keras.applications.DenseNet121(
            include_top=False, weights='imagenet', input_shape=INPUT_SHAPE, pooling='avg'
        )
    elif model_name == 'EfficientNetB0':
        base_model = tf.keras.applications.EfficientNetB0(
            include_top=False, weights='imagenet', input_shape=INPUT_SHAPE, pooling='avg'
        )
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    
    base_model.trainable = True
    
    # Add classification head
    inputs = base_model.input
    x = base_model.output
    x = tf.keras.layers.Dense(512, activation='relu', name=f'{model_name}_dense_512')(x)
    x = tf.keras.layers.Dropout(0.5, name=f'{model_name}_dropout')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name=f'{model_name}_output')(x)
    
    model = tf.keras.Model(inputs, outputs, name=f'{model_name}_finetuned')
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def train(args):
    # Load data
    train_files, train_labels = get_files_from_structure(args.train_path)
    val_files, val_labels = get_files_from_structure(args.val_path)
    
    # Create generators (using 'imagenet' preprocessing)
    train_gen = image_generator(train_files, train_labels, BATCH_SIZE, IMG_DIM, 'imagenet')
    val_gen = image_generator(val_files, val_labels, BATCH_SIZE, IMG_DIM, 'imagenet')

    for model_name in ['DenseNet121', 'EfficientNetB0']:
        print(f"\n************ TRAINING {model_name} MODEL ************")
        
        model = create_fine_tuned_model(model_name)
        
        checkpoint_filepath = os.path.join(args.output_path, f"best_model_{model_name.lower()}.keras")
        model_checkpoint_callback = ModelCheckpoint(
            filepath=checkpoint_filepath, 
            save_best_only=True, 
            monitor='val_accuracy', 
            mode='max',
            verbose=1
        )
        
        early_stopping_callback = EarlyStopping(
            monitor='val_accuracy',
            patience=5,
            restore_best_weights=True
        )

        model.fit(train_gen, 
                  epochs=args.epochs, 
                  steps_per_epoch=TRAIN_STEPS,
                  validation_data=val_gen,
                  validation_steps=VAL_STEPS,
                  callbacks=[model_checkpoint_callback, early_stopping_callback])
        
        print(f"Finished training. Best {model_name} model saved to {checkpoint_filepath}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description='Train Backbone Models (DenseNet, EfficientNet)')
    parser.add_argument('--train_path', type=str, required=True, help='Path to training data folder')
    parser.add_argument('--val_path', type=str, required=True, help='Path to validation data folder')
    parser.add_argument('--output_path', type=str, default="/kaggle/working/models/", help='Path to save models')
    parser.add_argument('--epochs', type=int, default=3, help='Number of epochs')
    
    args = parser.parse_args()
    
    os.makedirs(args.output_path, exist_ok=True)
    train(args)

Overwriting train_backbones.py


Cell 5b: Run Backbone Model Training
This executes the script to train both DenseNet121 and EfficientNetB0.

In [48]:
!python train_backbones.py \
    --train_path "/kaggle/input/1000-videos-split/1000_videos/train" \
    --val_path "/kaggle/input/1000-videos-split/1000_videos/validation" \
    --output_path "/kaggle/working/models/" \
    --epochs 30

2025-11-15 14:37:40.154726: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763217460.176955  217646 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763217460.183504  217646 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
Found 11633 images in /kaggle/input/1000-videos-split/1000_videos/train: 6028

🧬 Cell 6: Part 3 - Hybrid Feature Extraction
Now for the main part of your Plan.txt. We'll write a script to load all three trained models, convert them into feature extractors, and process all our data (train, val, test) into "super-vectors."

Cell 6a: extract_hybrid_features.py

In [49]:
%%writefile extract_hybrid_features.py
import os
import argparse
import numpy as np
import tensorflow as tf
from tqdm import tqdm
from model_attention import ModifiedBranch, MainBranch, Attention
from utils_hybrid import get_files_from_structure, load_and_prep_image

# --- Constants ---
BATCH_SIZE = 32
IMG_DIM = (299, 299)
MODEL_PATH = "/kaggle/working/models/"
OUTPUT_PATH = "/kaggle/working/features/"

# --- Helper Function ---
def load_extractors():
    print("Loading all three trained models...")
    
    # 1. Load Attn-Xception Model and create extractor
    custom_objects = {"ModifiedBranch": ModifiedBranch, "MainBranch": MainBranch, "Attention": Attention}
    model_attn = tf.keras.models.load_model(os.path.join(MODEL_PATH, "best_model_attention.keras"), custom_objects=custom_objects)
    
    # As per Plan.txt: get output of 'attention_output' layer and flatten it
    attn_output = model_attn.get_layer('attention_output').output
    flattened_attn = tf.keras.layers.Flatten()(attn_output)
    extractor_C = tf.keras.Model(inputs=model_attn.input, outputs=flattened_attn, name="Extractor_AttentionXception")
    print(f"Extractor C (Attention) loaded. Output shape: {extractor_C.output_shape}") # (None, 361)

    # 2. Load DenseNet121 and create extractor
    model_dense = tf.keras.models.load_model(os.path.join(MODEL_PATH, "best_model_densenet121.keras"))
    # Output is the GlobalAveragePooling layer, which is 4th from the end
    extractor_A = tf.keras.Model(inputs=model_dense.input, outputs=model_dense.layers[-4].output, name="Extractor_DenseNet")
    print(f"Extractor A (DenseNet) loaded. Output shape: {extractor_A.output_shape}") # (None, 1024)

    # 3. Load EfficientNetB0 and create extractor
    model_effnet = tf.keras.models.load_model(os.path.join(MODEL_PATH, "best_model_efficientnetb0.keras"))
    # Output is the GlobalAveragePooling layer
    extractor_B = tf.keras.Model(inputs=model_effnet.input, outputs=model_effnet.layers[-4].output, name="Extractor_EfficientNet")
    print(f"Extractor B (EfficientNet) loaded. Output shape: {extractor_B.output_shape}") # (None, 1280)

    return extractor_A, extractor_B, extractor_C

def extract_features_from_paths(paths, labels, extractors):
    extractor_A, extractor_B, extractor_C = extractors
    
    all_features = []
    all_labels = []
    
    num_batches = int(np.ceil(len(paths) / BATCH_SIZE))
    
    for i in tqdm(range(num_batches), desc="Extracting features"):
        batch_paths = paths[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
        batch_labels = labels[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
        
        batch_x_imagenet = [] # For DenseNet, EfficientNet
        batch_x_xception = [] # For Attention-Xception
        valid_labels = []
        
        for j, path in enumerate(batch_paths):
            # Load and prep for Imagenet (A, B)
            img_inet = load_and_prep_image(path, IMG_DIM, 'imagenet')
            # Load and prep for Xception (C)
            img_xcept = load_and_prep_image(path, IMG_DIM, 'xception')
            
            if img_inet is not None and img_xcept is not None:
                batch_x_imagenet.append(img_inet)
                batch_x_xception.append(img_xcept)
                valid_labels.append(batch_labels[j])
        
        if not batch_x_imagenet:
            continue
            
        # Convert to numpy arrays
        batch_x_imagenet = np.array(batch_x_imagenet)
        batch_x_xception = np.array(batch_x_xception)
        
        # Predict features
        features_A = extractor_A.predict(batch_x_imagenet, verbose=0) # (None, 1024)
        features_B = extractor_B.predict(batch_x_imagenet, verbose=0) # (None, 1280)
        features_C = extractor_C.predict(batch_x_xception, verbose=0)  # (None, 361)
        
        # Concatenate into "super-vector" (Plan.txt Step 630)
        # Total: 1024 + 1280 + 361 = 2665 features
        batch_features_stacked = np.concatenate([features_A, features_B, features_C], axis=1)
        
        all_features.append(batch_features_stacked)
        all_labels.append(np.array(valid_labels))
        
    return np.concatenate(all_features, axis=0), np.concatenate(all_labels, axis=0)

def main():
    parser = argparse.ArgumentParser(description='Extract Hybrid Features')
    parser.add_argument('--data_path', type=str, required=True, help='Base path to dataset (e.g., .../1000_videos/)')
    parser.add_argument('--output_path', type=str, default=OUTPUT_PATH, help='Path to save .npy files')
    
    args = parser.parse_args()
    os.makedirs(args.output_path, exist_ok=True)
    
    # 1. Load extractors
    extractors = load_extractors()
    
    # 2. Process Train, Val, and Test sets
    for split in ['train', 'validation', 'test']:
        print(f"\n--- Processing {split} data ---")
        split_path = os.path.join(args.data_path, split)
        paths, labels = get_files_from_structure(split_path)
        
        features, labels = extract_features_from_paths(paths, labels, extractors)
        
        # 3. Save to disk
        np.save(os.path.join(args.output_path, f"{split}_features.npy"), features)
        np.save(os.path.join(args.output_path, f"{split}_labels.npy"), labels)
        print(f"Saved {split} features ({features.shape}) and labels ({labels.shape})")

if __name__ == "__main__":
    main()

Overwriting extract_hybrid_features.py


Cell 6b: Run Feature Extraction
This script loads our three trained models and processes all images from the train, validation, and test directories, saving the 2665-dimension feature vectors as .npy files.

In [50]:
!python extract_hybrid_features.py \
    --data_path "/kaggle/input/1000-videos-split/1000_videos/" \
    --output_path "/kaggle/working/features/"

2025-11-15 15:17:45.608028: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763219865.629674  217918 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763219865.636257  217918 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
Loading all three trained models...
I0000 00:00:1763219870.367665  217918 gpu

🎯 Cell 7: Part 4 - Feature Selection & KNN Training
This script remains the same. Its job is to load the train and validation features, run the feature selection pipeline, and then train and save the final KNN model, scaler, and selected feature indices. It does not touch the test set.

Cell 7a: train_knn.py

In [51]:
%%writefile train_knn.py
import os
import argparse
import numpy as np
import joblib
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from sklearn.metrics import accuracy_score

# --- Constants ---
FEATURE_PATH = "/kaggle/working/features/"
MODEL_PATH = "/kaggle/working/models/"

# --- CONFIGURATION ---
# Set the number of samples to use for the heavy feature selection step.
# 3000-5000 is a good starting point.
# This will be much faster than using all 11,633 samples.
FEATURE_SELECTION_SUBSET_SIZE = 10000 
# ---------------------

# --- Feature Selection Functions ---

def relief_f_score(X, y, k=10):
    n_samples, n_features = X.shape
    feature_scores = np.zeros(n_features)
    
    # We are already using a subset, so k=10 should be fine
    # and we iterate over all samples in the subset.
    for i in range(n_samples):
        distances = np.sum((X - X[i]) ** 2, axis=1)
        nearest_indices = np.argsort(distances)[1:k+1]
        
        hits = [idx for idx in nearest_indices if y[idx] == y[i]]
        misses = [idx for idx in nearest_indices if y[idx] != y[i]]
        
        for j in range(n_features):
            if hits:
                hit_diff = np.mean([abs(X[i, j] - X[idx, j]) for idx in hits])
                feature_scores[j] -= hit_diff / k
            
            if misses:
                miss_diff = np.mean([abs(X[i, j] - X[idx, j]) for idx in misses])
                feature_scores[j] += miss_diff / k
    
    # Add epsilon to prevent divide-by-zero if scores are all 0
    return (feature_scores - np.min(feature_scores)) / (np.max(feature_scores) - np.min(feature_scores) + 1e-6)

def mrmr_score(X, y):
    # Relevance (MI with target)
    relevance = mutual_info_classif(X, y)
    
    # Redundancy (MI between features)
    n_features = X.shape[1]
    redundancy = np.zeros(n_features)
    for i in range(n_features):
        if i == 0: continue
        # Approx: avg MI with a random sample of other features to speed up
        compare_indices = np.random.choice(i, min(50, i), replace=False)
        redundancy[i] = np.mean([mutual_info_classif(X[:, [i, j]], y)[0] for j in compare_indices])
        
    mrmr_scores = relevance - redundancy
    return (mrmr_scores - np.min(mrmr_scores)) / (np.max(mrmr_scores) - np.min(mrmr_scores) + 1e-6)

def calculate_fitness(X_train_subset_scaled, y_train_subset, X_val_scaled, y_val, feature_indices, weight=0.9):
    """
    Calculates fitness by training on the TRAIN SUBSET and scoring on the VALIDATION set.
    """
    if len(feature_indices) == 0:
        return 0
    
    X_train_sel = X_train_subset_scaled[:, feature_indices]
    X_val_sel = X_val_scaled[:, feature_indices]
    
    temp_knn = KNeighborsClassifier(n_neighbors=5)
    temp_knn.fit(X_train_sel, y_train_subset)
    accuracy = temp_knn.score(X_val_sel, y_val)
    
    feature_ratio = len(feature_indices) / X_train_subset_scaled.shape[1]
    fitness = weight * accuracy + (1 - weight) * (1 - feature_ratio)
    return fitness, accuracy

def run_feature_selection(
    X_train_scaled, y_train, 
    X_train_subset_scaled, y_train_subset, 
    X_val_scaled, y_val, 
    tau=0.3, alpha_pct=0.1, beta_pct=0.1, max_iterations=250):
    
    print("Starting feature selection...")
    n_features = X_train_scaled.shape[1]
    
    # Step 1: Calculate filter scores using the SUBSET
    print(f"Calculating scores using a subset of {len(y_train_subset)} samples...")
    print("Calculating ReliefF scores...")
    relief_scores = relief_f_score(X_train_subset_scaled, y_train_subset)
    
    print("Calculating MI scores...")
    mi_scores = mutual_info_classif(X_train_subset_scaled, y_train_subset)
    mi_scores = (mi_scores - np.min(mi_scores)) / (np.max(mi_scores) - np.min(mi_scores) + 1e-6)
    
    print("Calculating mRMR scores...")
    # mrmr_scores = mrmr_score(X_train_subset_scaled, y_train_subset)
    # mrmr_scores = mrmr_score(X_train_subset_scaled, y_train_subset)
    mrmr_scores = np.zeros(n_features) # <-- ADD THIS LINE to just skip it and save a lot of time
    
    # Step 2: Combine scores
    # combined_scores = (relief_scores + mi_scores + mrmr_scores) / 3
    combined_scores = (relief_scores + mi_scores) / 2 # <-- EDIT THIS LINE for fast output
    
    # Step 3: Initial feature selection
    n_initial = int(tau * n_features)
    initial_indices = np.argsort(combined_scores)[-n_initial:]
    
    # Step 4: Inclusion-Exclusion
    print("Starting Inclusion-Exclusion optimization...")
    best_features = initial_indices.copy()
    
    # Fitness is checked against the full validation set
    best_fitness, best_acc = calculate_fitness(
        X_train_subset_scaled, y_train_subset, X_val_scaled, y_val, best_features
    )
    print(f"Initial solution: {len(best_features)} features, Fitness: {best_fitness:.4f}, Val Acc: {best_acc:.4f}")
    
    all_indices = np.arange(n_features)
    
    for iteration in range(max_iterations):
        current_features = best_features.copy()
        
        # Exclusion
        n_exclude = max(1, int(alpha_pct * len(current_features)))
        if len(current_features) > n_exclude:
            exclude_indices = np.random.choice(len(current_features), n_exclude, replace=False)
            current_features = np.delete(current_features, exclude_indices)
        
        # Inclusion
        remaining_features = np.setdiff1d(all_indices, current_features)
        if len(remaining_features) > 0:
            n_include = min(max(1, int(beta_pct * n_features)), len(remaining_features))
            remaining_scores = combined_scores[remaining_features]
            include_indices_local = np.argsort(remaining_scores)[-n_include:]
            include_indices = remaining_features[include_indices_local]
            current_features = np.concatenate([current_features, include_indices])
            current_features = np.unique(current_features)
        
        current_fitness, current_acc = calculate_fitness(
            X_train_subset_scaled, y_train_subset, X_val_scaled, y_val, current_features
        )
        
        if current_fitness > best_fitness:
            best_features = current_features.copy()
            best_fitness = current_fitness
            print(f"Iter {iteration+1}/{max_iterations}: New best! Features: {len(best_features)}, Fitness: {best_fitness:.4f}, Val Acc: {current_acc:.4f}")
            
    print(f"\nFeature selection complete. Selected {len(best_features)} features.")
    return best_features

def main():
    print("--- Loading Extracted Features (Train/Val) ---")
    X_train = np.load(os.path.join(FEATURE_PATH, "train_features.npy"))
    y_train = np.load(os.path.join(FEATURE_PATH, "train_labels.npy"))
    X_val = np.load(os.path.join(FEATURE_PATH, "validation_features.npy"))
    y_val = np.load(os.path.join(FEATURE_PATH, "validation_labels.npy"))
    
    print(f"Total Train features: {X_train.shape}, Total Val features: {X_val.shape}")
    
    # 1. Scale all data first
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # 2. Create balanced subset for feature selection (as you requested)
    n_samples = X_train.shape[0]
    subset_size = min(FEATURE_SELECTION_SUBSET_SIZE, n_samples)
    
    fake_indices = np.where(y_train == 1)[0]
    real_indices = np.where(y_train == 0)[0]
    
    n_fake = int(subset_size / 2)
    n_real = subset_size - n_fake
    
    # Handle potential class imbalance in subset size
    n_fake = min(n_fake, len(fake_indices))
    n_real = min(n_real, len(real_indices))
    
    subset_fake_indices = np.random.choice(fake_indices, n_fake, replace=False)
    subset_real_indices = np.random.choice(real_indices, n_real, replace=False)
    
    subset_indices = np.concatenate([subset_fake_indices, subset_real_indices])
    np.random.shuffle(subset_indices) # Shuffle the final subset indices
    
    X_train_subset_scaled = X_train_scaled[subset_indices]
    y_train_subset = y_train[subset_indices]
    
    print(f"Using a balanced subset of {len(y_train_subset)} samples for feature selection.")

    # 3. Run Feature Selection on the subset
    selected_indices = run_feature_selection(
        X_train_scaled, y_train, 
        X_train_subset_scaled, y_train_subset, 
        X_val_scaled, y_val,
        tau=0.3, alpha_pct=0.1, beta_pct=0.1
    )
    
    # Save selected feature indices
    np.save(os.path.join(MODEL_PATH, "selected_features.npy"), selected_indices)
    
    # 4. Train Final KNN Classifier on the FULL training set
    print("\n--- Training Final KNN Classifier on ALL Training Data ---")
    
    # Filter the FULL training set with selected features
    X_train_selected = X_train[:, selected_indices]
    
    # Create a new scaler and fit it ONLY on the selected training features
    final_scaler = StandardScaler()
    X_train_final_scaled = final_scaler.fit_transform(X_train_selected)
    
    # Train KNN
    knn_model = KNeighborsClassifier(n_neighbors=5)
    knn_model.fit(X_train_final_scaled, y_train)
    
    # 5. Save Final Scaler and KNN Model
    joblib.dump(final_scaler, os.path.join(MODEL_PATH, "scaler.joblib"))
    joblib.dump(knn_model, os.path.join(MODEL_PATH, "knn_model.joblib"))
    
    print(f"Final KNN model and Scaler saved to {MODEL_PATH}")

if __name__ == "__main__":
    main()

Overwriting train_knn.py


Cell 7b: Run Feature Selection and KNN Training

In [52]:
!python train_knn.py

--- Loading Extracted Features (Train/Val) ---
Total Train features: (11633, 2665), Total Val features: (2400, 2665)
Using a balanced subset of 10000 samples for feature selection.
Starting feature selection...
Calculating scores using a subset of 10000 samples...
Calculating ReliefF scores...
Calculating MI scores...
Calculating mRMR scores...
Starting Inclusion-Exclusion optimization...
Initial solution: 799 features, Fitness: 0.9138, Val Acc: 0.9375

Feature selection complete. Selected 799 features.

--- Training Final KNN Classifier on ALL Training Data ---
Final KNN model and Scaler saved to /kaggle/working/models/


📊 Cell 8: Part 5 - Scoring (Fast Evaluation)
This is the new script you requested. It's fast because it uses the pre-extracted test_features.npy file. It loads the knn_model.joblib, scaler.joblib, and selected_features.npy that we just saved and runs an evaluation on the test set.

In [53]:
%%writefile score_knn.py
import os
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix, 
    classification_report, f1_score
)

# --- Constants ---
FEATURE_PATH = "/kaggle/working/features/"
MODEL_PATH = "/kaggle/working/models/"

def evaluate():
    print("--- Loading Test Features and Saved Models ---")
    
    try:
        X_test = np.load(os.path.join(FEATURE_PATH, "test_features.npy"))
        y_test = np.load(os.path.join(FEATURE_PATH, "test_labels.npy"))
        
        knn_model = joblib.load(os.path.join(MODEL_PATH, "knn_model.joblib"))
        scaler = joblib.load(os.path.join(MODEL_PATH, "scaler.joblib"))
        selected_indices = np.load(os.path.join(MODEL_PATH, "selected_features.npy"))
    except FileNotFoundError as e:
        print(f"Error loading model files: {e}")
        print("Please ensure Part 4 (train_knn.py) ran successfully.")
        return

    print(f"Test features: {X_test.shape}, Test labels: {y_test.shape}")
    print(f"Loaded KNN model, Scaler, and {len(selected_indices)} selected features.")
    
    # 1. Select features from the test set
    X_test_selected = X_test[:, selected_indices]
    
    # 2. Scale the test data
    X_test_scaled = scaler.transform(X_test_selected)
    
    # 3. Make predictions
    print("\nRunning predictions on test set...")
    y_pred = knn_model.predict(X_test_scaled)
    y_proba = knn_model.predict_proba(X_test_scaled)[:, 1] # Probability of 'fake'
    
    # 4. Calculate and print metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    
    print("\n--- Test Set Evaluation Results ---")
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(f"AUC Score: {auc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['real (0)', 'fake (1)']))
    
    # 5. Plot Confusion Matrix
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['real', 'fake'], yticklabels=['real', 'fake'])
    plt.title('Confusion Matrix - Hybrid Model (Test Set)', fontsize=16)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.savefig("/kaggle/working/hybrid_confusion_matrix.png")
    print("\nConfusion matrix saved to /kaggle/working/hybrid_confusion_matrix.png")

if __name__ == "__main__":
    evaluate()

Overwriting score_knn.py


🚀 Cell 9: Part 6 - Final Prediction Script (for new images)
This is the script from my original Cell 8. We still need this for the final goal stated in your plan: "predict deepfake single image or directory".

This script is slow because it works on raw image files, not pre-extracted features. You use this after you are happy with your score from Part 5.

Cell 9a: predict_hybrid.py

In [54]:
%%writefile predict_hybrid.py
import os
import argparse
import numpy as np
import tensorflow as tf
import joblib
from tqdm import tqdm
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score  # <-- FIXED: Added accuracy_score
)
from model_attention import ModifiedBranch, MainBranch, Attention
from utils_hybrid import load_and_prep_image

# --- Constants ---
MODEL_PATH = "/kaggle/working/models/"
IMG_DIM = (299, 299)
LABELS_MAP = {0: 'real', 1: 'fake'}
LABELS_MAP_INV = {'real': 0, 'fake': 1}

# --- Global Models (Load once) ---
EXTRACTOR_A = None
EXTRACTOR_B = None
EXTRACTOR_C = None
KNN_MODEL = None
SCALER = None
SELECTED_INDICES = None

def load_all_models():
    global EXTRACTOR_A, EXTRACTOR_B, EXTRACTOR_C, KNN_MODEL, SCALER, SELECTED_INDICES
    
    print("Loading all models for prediction...")
    
    # 1. Load Attn-Xception
    custom_objects = {"ModifiedBranch": ModifiedBranch, "MainBranch": MainBranch, "Attention": Attention}
    model_attn = tf.keras.models.load_model(os.path.join(MODEL_PATH, "best_model_attention.keras"), custom_objects=custom_objects)
    attn_output = model_attn.get_layer('attention_output').output
    flattened_attn = tf.keras.layers.Flatten()(attn_output)
    EXTRACTOR_C = tf.keras.Model(inputs=model_attn.input, outputs=flattened_attn)

    # 2. Load DenseNet121
    model_dense = tf.keras.models.load_model(os.path.join(MODEL_PATH, "best_model_densenet121.keras"))
    EXTRACTOR_A = tf.keras.Model(inputs=model_dense.input, outputs=model_dense.layers[-4].output)

    # 3. Load EfficientNetB0
    model_effnet = tf.keras.models.load_model(os.path.join(MODEL_PATH, "best_model_efficientnetb0.keras"))
    EXTRACTOR_B = tf.keras.Model(inputs=model_effnet.input, outputs=model_effnet.layers[-4].output)
    
    print("DL extractors loaded.")

    # 4. Load KNN, Scaler, and Features
    KNN_MODEL = joblib.load(os.path.join(MODEL_PATH, "knn_model.joblib"))
    SCALER = joblib.load(os.path.join(MODEL_PATH, "scaler.joblib"))
    SELECTED_INDICES = np.load(os.path.join(MODEL_PATH, "selected_features.npy"))
    
    print("KNN, Scaler, and Feature Indices loaded. Ready to predict.")

def predict_single_image(image_path):
    # Load and prep images (one for each preprocess type)
    img_inet = load_and_prep_image(image_path, IMG_DIM, 'imagenet')
    img_xcept = load_and_prep_image(image_path, IMG_DIM, 'xception')
    
    if img_inet is None or img_xcept is None:
        print(f"Error: Could not process {image_path}")
        return "Error", 0.0
        
    # Add batch dimension
    batch_x_imagenet = np.expand_dims(img_inet, axis=0)
    batch_x_xception = np.expand_dims(img_xcept, axis=0)
    
    # 1. Extract features from all 3 models
    features_A = EXTRACTOR_A.predict(batch_x_imagenet, verbose=0)
    features_B = EXTRACTOR_B.predict(batch_x_imagenet, verbose=0)
    features_C = EXTRACTOR_C.predict(batch_x_xception, verbose=0)
    
    # 2. Create "super-vector"
    features_stacked = np.concatenate([features_A, features_B, features_C], axis=1) # (1, 2665)
    
    # 3. Filter with selected features
    features_selected = features_stacked[:, SELECTED_INDICES] # (1, num_selected)
    
    # 4. Scale
    features_scaled = SCALER.transform(features_selected)
    
    # 5. Predict with KNN
    prediction_proba = KNN_MODEL.predict_proba(features_scaled)
    
    predicted_index = np.argmax(prediction_proba[0])
    predicted_label = LABELS_MAP[predicted_index]
    confidence = prediction_proba[0][predicted_index] * 100
    
    return predicted_label, confidence

def evaluate_folder(folder_path):
    print(f"Scanning folder: {folder_path} (as requested in data structure)")
    
    fake_paths = glob.glob(os.path.join(folder_path, 'fake', '*.png'))
    real_paths = glob.glob(os.path.join(folder_path, 'real', '*.png'))
    
    all_paths = fake_paths + real_paths
    true_labels_str = ['fake'] * len(fake_paths) + ['real'] * len(real_paths)
    true_labels_int = [LABELS_MAP_INV[l] for l in true_labels_str]
    
    if not all_paths:
        print("No .png images found in 'fake' or 'real' subdirectories.")
        return
        
    all_preds_int = []
    
    for i, path in enumerate(tqdm(all_paths, desc="Evaluating image folder")):
        pred_label, _ = predict_single_image(path)
        all_preds_int.append(LABELS_MAP_INV[pred_label])
            
    accuracy = accuracy_score(true_labels_int, all_preds_int)
    
    print("\n--- Evaluation Summary ---")
    print(f"Total Images: {len(all_paths)}")
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("--------------------------")
    
    print("\nClassification Report:")
    print(classification_report(true_labels_int, all_preds_int, target_names=['real (0)', 'fake (1)']))
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(true_labels_int, all_preds_int)
    print(cm)
    
    # Plotting
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
                xticklabels=['real', 'fake'], yticklabels=['real', 'fake'])
    plt.title('Confusion Matrix - Hybrid Model (Raw Images)', fontsize=16)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.savefig("/kaggle/working/hybrid_confusion_matrix_raw_images.png")
    print("\nConfusion matrix saved to /kaggle/working/hybrid_confusion_matrix_raw_images.png")

def main():
    parser = argparse.ArgumentParser(description='Predict if an image is real or fake.')
    parser.add_argument('--input_path', type=str, required=True, help='Path to an image file OR a test folder (with fake/real subdirs).')
    args = parser.parse_args()
    
    # Load all models into memory
    load_all_models()
    
    if os.path.isfile(args.input_path):
        pred_label, confidence = predict_single_image(args.input_path)
        print("\n--- Prediction Result ---")
        print(f"       File: {os.path.basename(args.input_path)}")
        print(f"Prediction is: {pred_label.upper()}")
        print(f"  Confidence: {confidence:.2f}%")
        print("-------------------------")
        
    elif os.path.isdir(args.input_path):
        evaluate_folder(args.input_path)
    else:
        print(f"Error: Input path is not a valid file or directory: {args.input_path}")

if __name__ == "__main__":
    main()

Overwriting predict_hybrid.py


Cell 9b: Run Prediction on Test Folder (Slow) or Single Image
This cell now serves as your final prediction tool. You can point it at the test directory to get a final score (which will be slow, but it's a true end-to-end test) or point it at any single .png image you want to classify.

In [55]:
# # Run evaluation on the entire test set (from raw images, will be slow)
# !python predict_hybrid.py --input_path "/kaggle/input/1000-videos-split/1000_videos/test"
# Run evaluation on the entire test set (from raw images, will be slow)
!python predict_hybrid.py --input_path "/kaggle/input/1000-videos-split/1000_videos/test"
# Or, test a single file:
# !python predict_hybrid.py --input_path "/kaggle/input/1000-videos-split/1000_videos/test/fake/067_025_1.png"

2025-11-15 15:49:38.320469: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763221778.341387  236537 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763221778.347731  236537 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
Loading all models for prediction...
I0000 00:00:1763221783.219160  236537 gp